In [1]:
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [2]:
import glob
from ISRpy import beampack

jro0 = beampack.jrobeam

from IPython import display
import numpy as np

In [3]:
from OXmodeCython import OXmode_cython

In [42]:
dz=0.5 # resolution in km --- 0.1 is for good resolution, 0.5 for speed
zmax = 800
Z = np.arange(0.01,zmax,dz)

In [43]:
d = np.load('JM91J_2016014141141.npy', allow_pickle=True)[()]
freqs = d['vipir_freqs']
hts = d['vipir_hts']

In [44]:
def fps_parabolic(a,z,beta=7):
    def softplus(x):
        return 1/beta*np.log(1+np.exp(x*beta))
    para1 = a[0]-((z-a[1])/a[2]/3)**2
    para2 = a[3]-((z-a[4])/a[5]/3)**2
    para3 = a[6]-((z-a[7])/a[8]/3)**2

    fps1 = (80.6*(softplus(para1) + softplus(para2) + softplus(para3)))**0.5
    # fps2 = (80.6*np.max(np.vstack([para1,para2,para3]), axis=0))**0.5
    return fps1#,fps2


In [175]:
from matplotlib import pyplot as plt
from matplotlib import rcParams
import scipy.interpolate as inter
import numpy as np
import datetime
import scipy.constants as const
import time

from scipy.sparse import coo_matrix
from PIL import Image

def graph_generate(params):
    try:
        ppparam, fes, dz = params
        rcParams['figure.subplot.right'] = 0.78
        fig,axes = plt.subplots(1,3,figsize=(12.0,6.0),sharex=True)
        ax1 = axes[0]
        fps=fps_parabolic(ppparam, Z)
        ax1.plot(fps,Z)
        fvsO,vhsO,fvsX,vhsX=OXmode_cython(fps,fes,dz) # x1 y1 x2 y2 

        asym_fO = []
        asym_fX = []
        for i in range(1,fvsO.shape[0]):
            if vhsO[i] - vhsO[i-1] > 100:
                asym_fO += [fvsO[i]]
        for i in range(1,fvsX.shape[0]):
            if vhsX[i] - vhsX[i-1] > 100:
                asym_fX += [fvsX[i]]

        print(asym_fO, asym_fX)
        def Coo2Mat(fvs,vhs):
            ifvs, ivhs = [],[]
            for (fv,hv) in zip(fvs,vhs):
                if fv <= freqs.max() and fv >= freqs.min() and hv <= hts.max() and hv >= hts.min():
                    ifv = np.argmin(np.abs(freqs - fv))
                    ihv = np.argmin(np.abs(hts - hv))
                    ifvs += [ifv]
                    ivhs += [ihv]
            return ifvs, ivhs

        ifvsO, ivhsO = Coo2Mat(fvsO,vhsO)
        ifvsX, ivhsX = Coo2Mat(fvsX,vhsX)

        omode = coo_matrix((np.ones(len(ifvsO)), (ivhsO, ifvsO)), shape=(hts.shape[0], freqs.shape[0])).toarray()
        xmode = coo_matrix((np.ones(len(ifvsX)), (ivhsX, ifvsX)), shape=(hts.shape[0], freqs.shape[0])).toarray()

        def scatter_around_points(x_data : np.ndarray, y_data : np.ndarray, points=25):
            noisy_x = x_data.tolist()
            noisy_y = y_data.tolist()
            M = freqs.shape[0]
            N = hts.shape[0]
            for idx in range(x_data.shape[0]):
                if idx < x_data.shape[0]-1: 
                    sigma_y = abs(y_data[idx+1] - y_data[idx])
                else:
                    sigma_y = abs(y_data[idx] - y_data[idx-1])        

                for _ in range(points):  # add noise points
                    dx = np.random.uniform(-2,2)
                    dy = np.random.uniform(-1,1)*np.clip(sigma_y,1,15)
                    nx = x_data[idx] + dx
                    ny = y_data[idx] + dy
                    if nx >= 0 and nx < M and ny >=0 and ny < N:
                        noisy_x += [nx]
                        noisy_y += [ny]
                
                if sigma_y > 50:  # at asymptotic line at more noise points between two data points
                    num_points = int(points*sigma_y/10)
                    for _ in range(num_points):
                        dx = np.random.normal(0, 1)
                        dy = np.random.normal(0, 1) * 35
                        nx = x_data[idx] + dx
                        ny = y_data[idx] + sigma_y/2 + dy
                        if nx >= 0 and nx < M and ny >=0 and ny < N:
                            noisy_x += [nx]
                            noisy_y += [ny]
            return np.array(noisy_x), np.array(noisy_y)

        # # Generate noisy data
        fvsO_noisy_x, vhsO_noisy_y = scatter_around_points(np.array(ifvsO), np.array(ivhsO))
        fvsX_noisy_x, vhsX_noisy_y = scatter_around_points(np.array(ifvsX), np.array(ivhsX))
        
        omode = coo_matrix((np.ones(fvsO_noisy_x.shape), (vhsO_noisy_y, fvsO_noisy_x)), shape=(hts.shape[0], freqs.shape[0])).toarray()
        xmode = coo_matrix((np.ones(fvsX_noisy_x.shape), (vhsX_noisy_y, fvsX_noisy_x)), shape=(hts.shape[0], freqs.shape[0])).toarray()

        omode_data = np.zeros((*omode.shape,3), dtype=np.uint8)
        omode_data[:,:,0] = np.flip(omode>0,axis=0) * 255
        imgO = Image.fromarray(omode_data, 'RGB')

        xmode_data = np.zeros((*xmode.shape,3), dtype=np.uint8)
        xmode_data[:,:,1] = np.flip(xmode>0,axis=0) * 255
        imgX = Image.fromarray(omode_data, 'RGB')

        ax1.pcolormesh(freqs,hts,(omode+xmode)>0,shading='auto')
        axes[1].pcolormesh(freqs,hts,omode>0,shading='auto')
        for f in asym_fO: axes[1].plot([f,f],[0,700])
        axes[2].pcolormesh(freqs,hts,xmode>0,shading='auto')
        for f in asym_fX: axes[2].plot([f,f],[0,700])
        
        file_name = "_".join([f"{x:.5f}" for x in ppparam]) + "_.png"
        dir_name = 'training_data_parabola_fof2'
        fig.savefig(f"{dir_name}/visual/{file_name}", dpi=100, bbox_inches='tight', pad_inches=0, facecolor=fig.get_facecolor())
        fig.clear()
        
        imgTrain = Image.fromarray(np.flip((omode+xmode)>0,axis=0).astype(dtype=np.uint8)*255, 'L')
        imgTrain.save(f"{dir_name}/Train/{file_name}")
        imgO.save(f"{dir_name}/O/{file_name}")
        imgX.save(f"{dir_name}/X/{file_name}")

        np.save(f"{dir_name}/asym_f/{file_name}", {'asym_fO':asym_fO, 'asym_fX':asym_fX})

    except:
        # print(e)
        pass



In [177]:
dt0 = datetime.datetime(2015,4,15)
igrf_year = dt0.year + (dt0-datetime.datetime(dt0.year,1,1)).total_seconds() / (
        datetime.datetime(dt0.year+1,1,1)-datetime.datetime(dt0.year,1,1)).total_seconds()
dz=0.5 # resolution in km --- 0.1 is for good resolution, 0.5 for speed
zmax = 800
Z = np.arange(0.01,zmax,dz)
fes = []
for iZ in Z:
    r,geoc_lat,lon,aspect,B = jro0.aspect_angle(igrf_year,jro0.xyz0 + jro0.zenith0 * iZ)
    fes += [const.e * np.linalg.norm(B) * 1e-9 / const.m_e /1e6 /2/np.pi] #MHz
fes = np.array(fes)
param = [np.random.uniform(0.75, 1.25) * 0.9,
         np.random.uniform(300, 500),
         np.random.uniform(0.75, 1.25) * 60,  
         np.random.uniform(0.75, 1.25) * 0.2,  
         np.random.uniform(220,320),
         np.random.uniform(0.75, 1.25) * 70,
         np.random.uniform(0.75, 1.25) * 0.04,
         np.random.uniform(130,170),
         np.random.uniform(0.75, 1.25) * 20] 
# print(param, len(param))
graph_generate((param,fes,dz))

[4.725, 8.325] [5.075, 8.625]


<Figure size 1200x600 with 0 Axes>

In [178]:
from joblib import Parallel, delayed
import random 
import os
import multiprocessing as mp
from multiprocessing import Pool

# Generate parameters for the images [0.9,400,40,0.5,300,40,0.1,150,40]
parameters = []
for i in range(30000):  # Adjusted to generate 1000 sets of paramete
    igrf_year = dt0.year + (dt0-datetime.datetime(dt0.year,1,1)).total_seconds() / (datetime.datetime(dt0.year+1,1,1)-datetime.datetime(dt0.year,1,1)).total_seconds()
    param = [random.uniform(0.75, 1.25) * 0.9,
         random.uniform(300, 500),
         random.uniform(0.75, 1.25) * 60,  
         random.uniform(0.75, 1.25) * 0.2,  
         random.uniform(220,320),
         random.uniform(0.75, 1.25) * 70,
         random.uniform(0.75, 1.25) * 0.04,
         random.uniform(130,170),
         random.uniform(0.75, 1.25) * 20] 
    parameters += [(param,fes,dz)]

# Use all available CPU cores to generate the graphs in parallel
Parallel(n_jobs=mp.cpu_count()-8)(delayed(graph_generate)(p) for p in parameters)
# processes_pool = Pool(20)
# processes_pool.map(graph_generate, parameters)
# processes_pool.close()
# elapsed_time = time.time() - start_time
# print(f'Generated 1000 images. Time elapsed: {elapsed_time:.2f} seconds')



In [ ]:

dt0 = datetime.datetime(2015,4,15)
igrf_year = dt0.year + (dt0-datetime.datetime(dt0.year,1,1)).total_seconds() / (
        datetime.datetime(dt0.year+1,1,1)-datetime.datetime(dt0.year,1,1)).total_seconds()

dz=0.5 # resolution in km --- 0.1 is for good resolution, 0.5 for speed
zmin = 0
zmax = 800

rcParams['figure.subplot.right'] = 0.78
#set up a plot
fig,axes = plt.subplots(1,1,figsize=(9.0,8.0),sharex=True)
ax1 = axes

# height grid
Z = np.arange(0.01,zmax,dz)

# fps=fps_gauss(np.array([0.9,380,22,0.2,150,10]),Z)
# m, = ax1.plot (fps, Z, 'k-', label='spline fp')

fps1, fps2=fps_parabolic(np.array([0.9,400,40,0.5,300,40,0.1,150,40]),Z)
m, = ax1.plot (fps1, Z, 'r-', label='spline fp')
m, = ax1.plot (fps2, Z, 'b-', label='spline fp')

# start = time.time()
# fvsO,vhsO,fvsX,vhsX=OXmode_cython(fps,fes,dz)
# mO, = ax1.plot (fvsO, vhsO, 'g-', label='O-trace hv')
# mX, = ax1.plot (fvsX, vhsX, 'y-', label='X-trace hv')
# print('new oxmode:',time.time() - start)

# ax1.set_yscale('linear')
# ax1.set_xlim(0,12)
# ax1.set_ylim(0,zmax)
# ax1.set_xlabel('fp, fv (MHz)')
# ax1.set_ylabel('z, hv (km)')
# ax1.grid(True)
# ax1.yaxis.grid(True,which='minor',linestyle='--')
# ax1.legend(loc=2,prop={'size':11})
